In [2]:
from google.colab import files
import zipfile
import os
import shutil

# Upload northstar_dataset.zip
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_dir = "/content/northstar_dataset"

# Remove old folder if it already exists
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

# Extract ZIP
with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# Show extracted files
for root, dirs, files_list in os.walk(extract_dir):
    for file in files_list:
        print(os.path.join(root, file))

Saving northstar_dataset.zip to northstar_dataset.zip
/content/northstar_dataset/northstar_dataset/README.txt
/content/northstar_dataset/northstar_dataset/data_dictionary.csv
/content/northstar_dataset/northstar_dataset/drivers.csv
/content/northstar_dataset/northstar_dataset/incidents.csv
/content/northstar_dataset/northstar_dataset/customers.csv
/content/northstar_dataset/northstar_dataset/orders.csv
/content/northstar_dataset/northstar_dataset/complaints.csv
/content/northstar_dataset/northstar_dataset/deliveries.csv
/content/northstar_dataset/northstar_dataset/hubs.csv
/content/northstar_dataset/northstar_dataset/vehicles.csv
/content/northstar_dataset/northstar_dataset/app_events.csv


In [3]:
%load_ext rpy2.ipython

In [4]:
%%R

install.packages(c("readr", "dplyr", "ggplot2"))

library(readr)
library(dplyr)
library(ggplot2)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/readr_2.2.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.1.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/ggplot2_4.0.3.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpO9Dhz8/downloaded_packages’

Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



In [5]:
%%R

# Find all CSV files inside the extracted folder
csv_files <- list.files(
  "/content/northstar_dataset",
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

print(csv_files)

# Helper function to load CSV by file name
read_dataset <- function(file_name) {
  file_path <- csv_files[basename(csv_files) == file_name][1]
  read_csv(file_path, show_col_types = FALSE)
}

orders <- read_dataset("orders.csv")
deliveries <- read_dataset("deliveries.csv")
complaints <- read_dataset("complaints.csv")
incidents <- read_dataset("incidents.csv")
vehicles <- read_dataset("vehicles.csv")
app_events <- read_dataset("app_events.csv")

cat("Files loaded successfully")

 [1] "/content/northstar_dataset/northstar_dataset/app_events.csv"     
 [2] "/content/northstar_dataset/northstar_dataset/complaints.csv"     
 [3] "/content/northstar_dataset/northstar_dataset/customers.csv"      
 [4] "/content/northstar_dataset/northstar_dataset/data_dictionary.csv"
 [5] "/content/northstar_dataset/northstar_dataset/deliveries.csv"     
 [6] "/content/northstar_dataset/northstar_dataset/drivers.csv"        
 [7] "/content/northstar_dataset/northstar_dataset/hubs.csv"           
 [8] "/content/northstar_dataset/northstar_dataset/incidents.csv"      
 [9] "/content/northstar_dataset/northstar_dataset/orders.csv"         
[10] "/content/northstar_dataset/northstar_dataset/vehicles.csv"       
Files loaded successfully

In [6]:
%%R

# Create failed and delayed flags
deliveries <- deliveries %>%
  mutate(
    is_failed = ifelse(delivery_status == "Failed", 1, 0),
    is_delayed = ifelse(delivery_status == "Delayed", 1, 0),

    proof_missing = case_when(
      proof_of_completion_missing == TRUE ~ 1,
      proof_of_completion_missing == "TRUE" ~ 1,
      proof_of_completion_missing == "True" ~ 1,
      proof_of_completion_missing == "true" ~ 1,
      proof_of_completion_missing == 1 ~ 1,
      proof_of_completion_missing == "1" ~ 1,
      TRUE ~ 0
    ),

    override_group = case_when(
      manual_route_override_count == 0 ~ "No override",
      manual_route_override_count == 1 ~ "1 override",
      manual_route_override_count >= 2 ~ "2+ overrides",
      TRUE ~ "Unknown"
    )
  )

# Complaint count by order
complaint_counts <- complaints %>%
  group_by(order_id) %>%
  summarise(
    complaint_count = n(),
    total_compensation = sum(compensation_amount, na.rm = TRUE),
    .groups = "drop"
  )

# Incident count by delivery
incident_counts <- incidents %>%
  group_by(delivery_id) %>%
  summarise(
    incident_count = n(),
    .groups = "drop"
  )

# Create integrated main dataset
main <- deliveries %>%
  left_join(orders, by = "order_id") %>%
  left_join(vehicles, by = "vehicle_id") %>%
  left_join(complaint_counts, by = "order_id") %>%
  left_join(incident_counts, by = "delivery_id") %>%
  mutate(
    complaint_count = ifelse(is.na(complaint_count), 0, complaint_count),
    incident_count = ifelse(is.na(incident_count), 0, incident_count),
    total_compensation = ifelse(is.na(total_compensation), 0, total_compensation),
    has_complaint = ifelse(complaint_count > 0, 1, 0),
    has_incident = ifelse(incident_count > 0, 1, 0),
    direct_contribution_proxy = order_value - fuel_or_charge_cost - total_compensation
  )

cat("Main dataframe created successfully\n")
cat("Number of rows:", nrow(main), "\n")
cat("Number of columns:", ncol(main), "\n")

Main dataframe created successfully
Number of rows: 950 
Number of columns: 40 


In [7]:
%%R

required_columns <- c(
  "proof_missing",
  "is_failed",
  "maintenance_status",
  "delivery_status",
  "override_group",
  "manual_route_override_count"
)

required_columns %in% names(main)

names(main)

 [1] "delivery_id"                   "order_id"                     
 [3] "driver_id"                     "vehicle_id"                   
 [5] "hub_id"                        "dispatch_time"                
 [7] "delivery_completed_at"         "delivery_status"              
 [9] "route_distance_km"             "manual_route_override_count"  
[11] "proof_of_completion_missing"   "customer_rating_post_delivery"
[13] "fuel_or_charge_cost"           "is_failed"                    
[15] "is_delayed"                    "proof_missing"                
[17] "override_group"                "customer_id"                  
[19] "service_type"                  "order_created_at"             
[21] "promised_window_hours"         "pickup_zone"                  
[23] "dropoff_zone"                  "priority_level"               
[25] "order_value"                   "booking_channel"              
[27] "special_handling_flag"         "vehicle_type"                 
[29] "assigned_zone"              

In [8]:
%%R

# 1. Chi-square test: proof missing vs failed delivery
proof_table <- table(main$proof_missing, main$is_failed)
print(proof_table)

proof_test <- chisq.test(proof_table)
print(proof_test)


# 2. Odds ratio: missing proof and failure
a <- proof_table["1", "1"]  # proof missing and failed
b <- proof_table["1", "0"]  # proof missing and not failed
c <- proof_table["0", "1"]  # proof present and failed
d <- proof_table["0", "0"]  # proof present and not failed

odds_ratio <- (a / b) / (c / d)
cat("Odds Ratio:", round(odds_ratio, 2), "\n")


# 3. Chi-square test: maintenance status vs delivery outcome
maintenance_table <- table(main$maintenance_status, main$delivery_status)
print(maintenance_table)

maintenance_test <- chisq.test(maintenance_table)
print(maintenance_test)


# 4. Chi-square test: route override group vs delivery status
override_table <- table(main$override_group, main$delivery_status)
print(override_table)

override_test <- chisq.test(override_table)
print(override_test)


# 5. Spearman correlation: route override count vs failure flag
spearman_test <- cor.test(
  main$manual_route_override_count,
  main$is_failed,
  method = "spearman"
)

print(spearman_test)

   
      0   1
  0 773 108
  1  45  24

	Pearson's Chi-squared test with Yates' continuity correction

data:  proof_table
X-squared = 25.283, df = 1, p-value = 4.949e-07

Odds Ratio: 3.82 
           
            Delayed Failed OnTime
  Active        113     45    384
  InRepair       52     77    125
  Scheduled      37     10    107

	Pearson's Chi-squared test

data:  maintenance_table
X-squared = 81.325, df = 4, p-value < 2.2e-16

              
               Delayed Failed OnTime
  1 override        67     51    192
  2+ overrides      57     35    149
  No override       78     46    275

	Pearson's Chi-squared test

data:  override_table
X-squared = 6.1167, df = 4, p-value = 0.1906


	Spearman's rank correlation rho

data:  main$manual_route_override_count and main$is_failed
S = 136650877, p-value = 0.1784
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.0437018 



In addition: Warning message:
In cor.test.default(main$manual_route_override_count, main$is_failed,  :
  cannot compute exact p-value with ties


In [9]:
%%R

stat_summary <- data.frame(
  Test = c(
    "Chi-square: proof missing vs failed delivery",
    "Odds ratio: missing proof and failure",
    "Chi-square: maintenance status vs delivery outcome",
    "Chi-square: route override group vs delivery status",
    "Spearman correlation: override count vs failure flag"
  ),
  Statistic = c(
    round(proof_test$statistic, 2),
    round(odds_ratio, 2),
    round(maintenance_test$statistic, 2),
    round(override_test$statistic, 2),
    round(spearman_test$estimate, 2)
  ),
  P_value = c(
    signif(proof_test$p.value, 3),
    NA,
    signif(maintenance_test$p.value, 3),
    signif(override_test$p.value, 3),
    signif(spearman_test$p.value, 3)
  )
)

print(stat_summary)

                                                  Test Statistic  P_value
1         Chi-square: proof missing vs failed delivery     25.28 4.95e-07
2                Odds ratio: missing proof and failure      3.82       NA
3   Chi-square: maintenance status vs delivery outcome     81.32 9.13e-17
4  Chi-square: route override group vs delivery status      6.12 1.91e-01
5 Spearman correlation: override count vs failure flag      0.04 1.78e-01
